<a href="https://colab.research.google.com/github/shreyaghora/Bank_Marketing_Project/blob/main/Code/SMOTE.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install imblearn
!pip install xgboost
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import make_scorer, accuracy_score, precision_score, recall_score, f1_score, roc_auc_score
import warnings
warnings.filterwarnings('ignore')
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier, BaggingClassifier
from sklearn.svm import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import cohen_kappa_score
!pip install lightgbm
from lightgbm import LGBMClassifier
from sklearn.model_selection import RandomizedSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

# Install CatBoost
!pip install catboost

from catboost import CatBoostClassifier


from google.colab import files
uploaded = files.upload()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.7 MB/s eta 0:00:00


Saving bank-additional-full.csv to bank-additional-full.csv


In [ ]:
df = pd.read_csv('/content/bank-additional-full.csv', sep=';')

In [ ]:
# 3. Drop Leakage Feature
df.drop('duration', axis=1, inplace=True)


In [ ]:
# 4. Handle "unknown" Values
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].replace('unknown', df[col].mode()[0])

In [ ]:
# 5. Encode Target
df['y'] = df['y'].map({'no': 0, 'yes': 1})
# 6. One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)
X_shuffled = df.drop('y', axis=1)
y_shuffled = df['y']

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.linear_model import LogisticRegression
from scipy.stats import uniform

# Split
X_train, X_test, y_train, y_test = train_test_split(X_shuffled, y_shuffled, test_size=0.2, random_state=42)


In [ ]:
#Smote(Sampling) Techniques
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

In [ ]:
# Model
model =LogisticRegression(
         max_iter=1000,
         class_weight='balanced',
         random_state=42,
         C=1.0
      )

# Search space
param_dist = {
    'C':[0.6,0.8,1.0],
    'max_iter':[200,600,800]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(model, param_distributions=param_dist, n_iter=50,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'max_iter': 200, 'C': 0.8}
Best CV accuracy: 0.6310
Test accuracy: 0.8167


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Logistic Regression)
from sklearn.linear_model import LogisticRegression

clf_name = "Logistic Regression"
clf = LogisticRegression(random_state=42, max_iter=1000)



# Store results
results = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results_df = pd.DataFrame(results)

# Print
print(f"Results for {clf_name}:")
print(results_df)

# Save to CSV if needed
results_df.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Logistic Regression:
   Fold           Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Logistic Regression  0.900218   0.658385  0.229437     0.984960   
1     2  Logistic Regression  0.894149   0.627586  0.192389     0.985189   
2     3  Logistic Regression  0.906531   0.734694  0.237885     0.989359   
3     4  Logistic Regression  0.898034   0.632530  0.226293     0.983311   
4     5  Logistic Regression  0.899247   0.623529  0.231947     0.982523   
5     6  Logistic Regression  0.893421   0.701863  0.224206     0.986722   
6     7  Logistic Regression  0.906531   0.653061  0.223256     0.986175   
7     8  Logistic Regression  0.907016   0.704403  0.250000     0.987197   
8     9  Logistic Regression  0.901408   0.704225  0.215517     0.988506   
9    10  Logistic Regression  0.897523   0.715278  0.212810     0.988718   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.340289  0.475380  0.015040  0.607199  0.349102  0.299691   


In [ ]:
# Model
from sklearn.tree import DecisionTreeClassifier
model1 =DecisionTreeClassifier()

# Search space
param_dist1 = {

            'max_depth': [10,20,30],
            'min_samples_split': [20,30,40],
            'min_samples_leaf': [1,2,3]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# DecisionTreeClassifier

random_search = RandomizedSearchCV(model1, param_distributions=param_dist1, n_iter=50,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'min_samples_split': 40, 'min_samples_leaf': 1, 'max_depth': 10}
Best CV accuracy: 0.4072
Test accuracy: 0.8963


In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Decision Tree)
from sklearn.tree import DecisionTreeClassifier
clf_name = "Decision Tree"
clf = DecisionTreeClassifier()



# Store results
results1 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results1.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results1_df1 = pd.DataFrame(results1)

# Print
print(f"Results for {clf_name}:")
print(results1_df1)

# Save to CSV if needed
results1_df1.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Decision Tree:
   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Decision Tree  0.841224   0.310277  0.339827     0.904567  0.324380   
1     2  Decision Tree  0.834183   0.289157  0.304440     0.902907  0.296601   
2     3  Decision Tree  0.856033   0.347253  0.348018     0.918963  0.347635   
3     4  Decision Tree  0.835397   0.292636  0.325431     0.900137  0.308163   
4     5  Decision Tree  0.839767   0.311688  0.367615     0.898689  0.337349   
5     6  Decision Tree  0.843166   0.356275  0.349206     0.912033  0.352705   
6     7  Decision Tree  0.850206   0.303158  0.334884     0.910274  0.318232   
7     8  Decision Tree  0.841466   0.299413  0.341518     0.902479  0.319082   
8     9  Decision Tree  0.837057   0.302857  0.342672     0.899836  0.321537   
9    10  Decision Tree  0.837542   0.328386  0.365702     0.900385  0.346041   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.554433  0.095

In [ ]:
# Model
from sklearn.ensemble import RandomForestClassifier
model2 = RandomForestClassifier()
# Search space
param_dist2 = {
        'max_depth': [None,2],
        'min_samples_split': [2,4],
        'min_samples_leaf': [1,2]
}

In [ ]:

# RandomForestClassifier

random_search = RandomizedSearchCV(model2, param_distributions=param_dist2, n_iter=15,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'min_samples_split': 2, 'min_samples_leaf': 2, 'max_depth': 2}
Best CV accuracy: 0.8500
Test accuracy: 0.8895


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Random Forest)
from sklearn.ensemble import RandomForestClassifier
clf_name = "Random Forest"
clf = RandomForestClassifier()



# Store results
results2 = []

# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results2.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results2_df2 = pd.DataFrame(results2)

# Print
print(f"Results for {clf_name}:")
print(results2_df2)

# Save to CSV if needed
results2_df2.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Random Forest:
   Fold     Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1  Random Forest  0.890750   0.521739  0.311688     0.963905  0.390244   
1     2  Random Forest  0.887837   0.522822  0.266385     0.968459  0.352941   
2     3  Random Forest  0.895606   0.552632  0.277533     0.972169  0.369501   
3     4  Random Forest  0.885166   0.482890  0.273707     0.962791  0.349381   
4     5  Random Forest  0.888565   0.496212  0.286652     0.963681  0.363384   
5     6  Random Forest  0.884924   0.557692  0.287698     0.968188  0.379581   
6     7  Random Forest  0.898034   0.523364  0.260465     0.972350  0.347826   
7     8  Random Forest  0.898034   0.554688  0.316964     0.968946  0.403409   
8     9  Random Forest  0.887810   0.504202  0.258621     0.967707  0.341880   
9    10  Random Forest  0.889509   0.552727  0.314050     0.966153  0.400527   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.548122  0.036

In [ ]:
# Model
from sklearn.neighbors import KNeighborsClassifier
model3 = KNeighborsClassifier(
         n_neighbors=15,
         weights='distance')

# Search space
param_dist3 = {
         "n_neighbors": [3, 5, 7,15],
            "weights": ["uniform", "distance"]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# KNeighbors Classifier

random_search = RandomizedSearchCV(model3, param_distributions=param_dist3, n_iter=25,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'weights': 'uniform', 'n_neighbors': 15}
Best CV accuracy: 0.8123
Test accuracy: 0.8941


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (KNN)
from sklearn.neighbors import KNeighborsClassifier
clf_name = "KNN"
clf = KNeighborsClassifier()

# Store results
results3 = []




# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results3.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results3_df3 = pd.DataFrame(results3)

# Print
print(f"Results for {clf_name}:")
print(results3_df3)

# Save to CSV if needed
results3_df3.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for KNN:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1        KNN  0.889779   0.515625  0.285714     0.966092  0.367688   
1     2        KNN  0.882739   0.480469  0.260042     0.963522  0.337449   
2     3        KNN  0.896091   0.554622  0.290749     0.971078  0.381503   
3     4        KNN  0.892935   0.545098  0.299569     0.968263  0.386648   
4     5        KNN  0.886137   0.477778  0.282276     0.961496  0.354883   
5     6        KNN  0.881767   0.531599  0.283730     0.965145  0.369987   
6     7        KNN  0.894635   0.491453  0.267442     0.967742  0.346386   
7     8        KNN  0.895606   0.533835  0.316964     0.966222  0.397759   
8     9        KNN  0.885381   0.484252  0.265086     0.964149  0.342618   
9    10        KNN  0.887324   0.537037  0.299587     0.965603  0.384615   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.525382  0.033908  0.625903  0.329131  0.312717           0.625903  

In [ ]:
# Model
from sklearn.svm import LinearSVC
model4 = LinearSVC()



from sklearn.svm import LinearSVC

model4 = LinearSVC(random_state=42, C=1.0,tol=1e-3, max_iter=2000)

param_dist4 = {

        'loss':['hinge','squared_hinge'],
        'penalty':['l2']
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# LinearSVC

random_search = RandomizedSearchCV(model4, param_distributions=param_dist4, n_iter=10,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'penalty': 'l2', 'loss': 'squared_hinge'}
Best CV accuracy: 0.8862
Test accuracy: 0.8965


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (LinearSVC)
from sklearn.svm import LinearSVC
clf_name = "SVM"
clf = LinearSVC()


# Store results
results4 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results4.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results4_df4 = pd.DataFrame(results4)

# Print
print(f"Results for {clf_name}:")
print(results4_df4)

# Save to CSV if needed
results4_df4.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for SVM:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1        SVM  0.900461   0.694030  0.201299     0.988789  0.312081   
1     2        SVM  0.892935   0.619403  0.175476     0.986012  0.273476   
2     3        SVM  0.904346   0.727273  0.211454     0.990177  0.327645   
3     4        SVM  0.899005   0.673913  0.200431     0.987688  0.308970   
4     5        SVM  0.899005   0.647482  0.196937     0.986619  0.302013   
5     6        SVM  0.893178   0.725352  0.204365     0.989212  0.318885   
6     7        SVM  0.904831   0.663793  0.179070     0.989428  0.282051   
7     8        SVM  0.905317   0.741667  0.198661     0.991555  0.313380   
8     9        SVM  0.899951   0.728070  0.178879     0.991516  0.287197   
9    10        SVM  0.897280   0.752066  0.188017     0.991745  0.300826   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.446141  0.011211  0.595044  0.338119  0.275542           0.595044  

In [ ]:


# Model

from xgboost import XGBClassifier

model5 =  XGBClassifier(
         random_state=42,
         n_estimators=100,
         max_depth=3,
         learning_rate=0.3,
      )

param_dist5 = {
            'learning_rate': [0.1,0.3],
            'max_depth': [2,3],
            'n_estimators': [100,200,300]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# XGBoost

random_search = RandomizedSearchCV(model5, param_distributions=param_dist5, n_iter=100,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 100, 'max_depth': 2, 'learning_rate': 0.1}
Best CV accuracy: 0.6721
Test accuracy: 0.8978


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (XGBoost)
from xgboost import XGBClassifier
clf_name = "XGBoost"
clf =  XGBClassifier(use_label_encoder=False, eval_metric='logloss')


# Store results
results5 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results5.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results5_df5 = pd.DataFrame(results5)

# Print
print(f"Results for {clf_name}:")
print(results5_df5)

# Save to CSV if needed
results5_df5.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for XGBoost:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1    XGBoost  0.898034   0.592105  0.292208     0.974569  0.391304   
1     2    XGBoost  0.892450   0.575000  0.243129     0.976687  0.341753   
2     3    XGBoost  0.904588   0.644550  0.299559     0.979536  0.409023   
3     4    XGBoost  0.899490   0.608696  0.301724     0.975376  0.403458   
4     5    XGBoost  0.893906   0.544643  0.266958     0.972146  0.358297   
5     6    XGBoost  0.889051   0.606335  0.265873     0.975934  0.369655   
6     7    XGBoost  0.904346   0.597826  0.255814     0.979940  0.358306   
7     8    XGBoost  0.902889   0.612150  0.292411     0.977390  0.395770   
8     9    XGBoost  0.894609   0.572115  0.256466     0.975643  0.354167   
9    10    XGBoost  0.897037   0.625000  0.309917     0.975234  0.414365   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.533645  0.025431  0.633389  0.368158  0.342572           0.6333

In [ ]:


# Model LightGBM



model6 =  XGBClassifier(
         max_depth=[3,5],
         learning_rate=0.3,
         n_estimators=100
     )

param_dist6 = {
        'max_depth': [3,5],
        'learning_rate': [0.1,0.3],
        'n_estimators': [100,200]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# LightGBM

random_search = RandomizedSearchCV(model6, param_distributions=param_dist6, n_iter=100,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 100, 'max_depth': 3, 'learning_rate': 0.1}
Best CV accuracy: 0.5432
Test accuracy: 0.8997


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here
from xgboost import XGBClassifier
clf_name = "LightGBM"
clf =   XGBClassifier()


# Store results
results6 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results6.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results6_df6 = pd.DataFrame(results6)

# Print
print(f"Results for {clf_name}:")
print(results6_df6)

# Save to CSV if needed
results6_df6.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for LightGBM:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1   LightGBM  0.898034   0.592105  0.292208     0.974569  0.391304   
1     2   LightGBM  0.892450   0.575000  0.243129     0.976687  0.341753   
2     3   LightGBM  0.904588   0.644550  0.299559     0.979536  0.409023   
3     4   LightGBM  0.899490   0.608696  0.301724     0.975376  0.403458   
4     5   LightGBM  0.893906   0.544643  0.266958     0.972146  0.358297   
5     6   LightGBM  0.889051   0.606335  0.265873     0.975934  0.369655   
6     7   LightGBM  0.904346   0.597826  0.255814     0.979940  0.358306   
7     8   LightGBM  0.902889   0.612150  0.292411     0.977390  0.395770   
8     9   LightGBM  0.894609   0.572115  0.256466     0.975643  0.354167   
9    10   LightGBM  0.897037   0.625000  0.309917     0.975234  0.414365   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.533645  0.025431  0.633389  0.368158  0.342572           0.633

In [ ]:


# Model CatBoost

from xgboost import XGBClassifier

model7 =   XGBClassifier(n_estimators=500,
        max_depth=3,
        learning_rate=0.1,)

param_dist7 = {
     'n_estimators': [200,350,500],
      'max_depth': [1,3,5],
      'learning_rate': [0.1,0.3]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
random_search = RandomizedSearchCV(model7, param_distributions=param_dist7, n_iter=100,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 200, 'max_depth': 1, 'learning_rate': 0.1}
Best CV accuracy: 0.7732
Test accuracy: 0.8975


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time


from catboost import CatBoostClassifier
clf_name = "CatBoost"
clf =   CatBoostClassifier(
        n_estimators=500,
        max_depth=3,
        learning_rate=0.1,
    )
param_dist= {
        'n_estimators': [200,400,500],
        'max_depth': [1,3],
        'learning_rate': [0.1,0.2,0.3]

}
# Store results
results7 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results7.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results7_df7 = pd.DataFrame(results7)

# Print
print(f"Results for {clf_name}:")
print(results7_df7)

# Save to CSV if needed
results7_df7.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Streaming output truncated to the last 5000 lines.
36:	learn: 0.2766373	total: 575ms	remaining: 7.2s
37:	learn: 0.2763623	total: 583ms	remaining: 7.09s
38:	learn: 0.2761311	total: 592ms	remaining: 7s
39:	learn: 0.2759508	total: 601ms	remaining: 6.92s
40:	learn: 0.2757406	total: 609ms	remaining: 6.82s
41:	learn: 0.2756568	total: 616ms	remaining: 6.71s
42:	learn: 0.2755558	total: 624ms	remaining: 6.63s
43:	learn: 0.2753533	total: 632ms	remaining: 6.55s
44:	learn: 0.2752637	total: 640ms	remaining: 6.47s
45:	learn: 0.2752150	total: 658ms	remaining: 6.5s
46:	learn: 0.2751244	total: 670ms	remaining: 6.46s
47:	learn: 0.2750080	total: 682ms	remaining: 6.42s
48:	learn: 0.2748751	total: 704ms	remaining: 6.48s
49:	learn: 0.2747629	total: 722ms	remaining: 6.5s
50:	learn: 0.2745612	total: 730ms	remaining: 6.43s
51:	learn: 0.2744622	total: 738ms	remaining: 6.36s
52:	learn: 0.2744177	total: 746ms	remaining: 6.29s
53:	learn: 0.2743120	total: 755ms	remaining: 6.23s
54:	learn: 0.2742685	total: 763ms	rem

In [ ]:

# Model Gradient Boosting

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, BaggingClassifier, StackingClassifier

model8 =   XGBClassifier(
         random_state=42,
         learning_rate=0.3,
         n_estimators=20
     )

param_dist8 = {
            'n_estimators': [20,30,50],
            'learning_rate': [0.1,0.3]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# Gradient Boosting

random_search = RandomizedSearchCV(model8, param_distributions=param_dist8, n_iter=200,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'n_estimators': 20, 'learning_rate': 0.1}
Best CV accuracy: 0.4950
Test accuracy: 0.8980


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (Gradient Boosting)
from xgboost import  XGBClassifier
clf_name = "Gradient Boosting"
clf =    XGBClassifier()

# Store results
results8 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results8.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results8_df8= pd.DataFrame(results8)

# Print
print(f"Results for {clf_name}:")
print(results8_df8)

# Save to CSV if needed
results8_df8.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for Gradient Boosting:
   Fold         Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  Gradient Boosting  0.898034   0.592105  0.292208     0.974569   
1     2  Gradient Boosting  0.892450   0.575000  0.243129     0.976687   
2     3  Gradient Boosting  0.904588   0.644550  0.299559     0.979536   
3     4  Gradient Boosting  0.899490   0.608696  0.301724     0.975376   
4     5  Gradient Boosting  0.893906   0.544643  0.266958     0.972146   
5     6  Gradient Boosting  0.889051   0.606335  0.265873     0.975934   
6     7  Gradient Boosting  0.904346   0.597826  0.255814     0.979940   
7     8  Gradient Boosting  0.902889   0.612150  0.292411     0.977390   
8     9  Gradient Boosting  0.894609   0.572115  0.256466     0.975643   
9    10  Gradient Boosting  0.897037   0.625000  0.309917     0.975234   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.391304  0.533645  0.025431  0.633389  0.368158  0.342572   
1  0.341753  0.487300  0

In [ ]:


# Model MLP

from sklearn.neural_network import MLPClassifier
model9 =   MLPClassifier(
         random_state=42,
         hidden_layer_sizes=(64,32),
         activation='relu',
         solver='adam',
         max_iter=200,
         early_stopping=True
         )
param_dist9 = {
      'hidden_layer_sizes': [(32, 16),(64,32)],
          'activation': ['relu']
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# MLP

random_search = RandomizedSearchCV(model9, param_distributions=param_dist9, n_iter=300,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search.fit(X_train, y_train)

best_model_LR = random_search.best_estimator_
print("Best parameters:", random_search.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search.best_score_))
print("Test accuracy: {:.4f}".format(best_model_LR.score(X_test, y_test)))

Best parameters: {'hidden_layer_sizes': (64, 32), 'activation': 'relu'}
Best CV accuracy: 0.8473
Test accuracy: 0.8961


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (MLP)
from sklearn.neural_network import MLPClassifier
clf_name = "MLP"
clf =   MLPClassifier(max_iter=500)


# Store results
results9 = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results9.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results9_df9= pd.DataFrame(results9)

# Print
print(f"Results for {clf_name}:")
print(results9_df9)

# Save to CSV if needed
results9_df9.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for MLP:
   Fold Classifier  Accuracy  Precision    Recall  Specificity        F1  \
0     1        MLP  0.888080   0.666667  0.004329     0.999727  0.008602   
1     2        MLP  0.862102   0.417391  0.507400     0.908118  0.458015   
2     3        MLP  0.889779   0.000000  0.000000     1.000000  0.000000   
3     4        MLP  0.897548   0.650000  0.196121     0.986594  0.301325   
4     5        MLP  0.889051   0.000000  0.000000     1.000000  0.000000   
5     6        MLP  0.879825   0.519313  0.240079     0.969018  0.328358   
6     7        MLP  0.884681   0.432836  0.337209     0.948496  0.379085   
7     8        MLP  0.891964   0.503979  0.424107     0.949060  0.460606   
8     9        MLP  0.863283   0.418182  0.545259     0.903667  0.473340   
9    10        MLP  0.896309   0.688742  0.214876     0.987067  0.327559   

         GM       FPR       AUC       MCC     Kappa  Balanced Accuracy  \
0  0.065786  0.000273  0.502028  0.047439  0.007165           0.502028  

In [ ]:
# Model Bagging
from sklearn.ensemble import BaggingClassifier
model_bagging =   BaggingClassifier(
         random_state=42,
         n_estimators=200,
         max_samples=0.8,
         max_features=0.8
         )
param_dist10 = {
     'n_estimators': [50,100,200],
      'max_samples':[0.6,0.8,1.0]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# BaggingClassifier

random_search_bagging = RandomizedSearchCV(model_bagging, param_distributions=param_dist10, n_iter=10,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search_bagging.fit(X_train, y_train)

best_model_bagging = random_search_bagging.best_estimator_
print("Best parameters:", random_search_bagging.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search_bagging.best_score_))
print("Test accuracy: {:.4f}".format(best_model_bagging.score(X_test, y_test)))

Best parameters: {'n_estimators': 50, 'max_samples': 0.6}
Best CV accuracy: 0.3698
Test accuracy: 0.8968


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (BaggingClassifier)
from sklearn.ensemble import BaggingClassifier
clf_name = "BaggingClassifier"
clf_bagging =   BaggingClassifier()


# Store results
results_bagging = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf_bagging.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf_bagging.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results_bagging.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results_bagging_df= pd.DataFrame(results_bagging)

# Print
print(f"Results for {clf_name}:")
print(results_bagging_df)

# Save to CSV if needed
results_bagging_df.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for BaggingClassifier:
   Fold         Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  BaggingClassifier  0.898519   0.587302  0.320346     0.971561   
1     2  BaggingClassifier  0.891964   0.563636  0.262156     0.973670   
2     3  BaggingClassifier  0.899005   0.589623  0.275330     0.976262   
3     4  BaggingClassifier  0.895849   0.577093  0.282328     0.973735   
4     5  BaggingClassifier  0.891964   0.524000  0.286652     0.967504   
5     6  BaggingClassifier  0.889294   0.604348  0.275794     0.974827   
6     7  BaggingClassifier  0.903860   0.589474  0.260465     0.978856   
7     8  BaggingClassifier  0.902889   0.605263  0.308036     0.975484   
8     9  BaggingClassifier  0.894366   0.570732  0.252155     0.975917   
9    10  BaggingClassifier  0.896795   0.622407  0.309917     0.974959   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.414566  0.557885  0.028439  0.645954  0.384363  0.364229   
1  0.357864  0.505226  0

In [ ]:
# Model Stacking
from sklearn.ensemble import StackingClassifier
model_stacking =   StackingClassifier(
          cv=10,
        estimators=[
            ('lr', LogisticRegression(random_state=42)),
            ('rf', RandomForestClassifier(random_state=42)),
            ('dt', DecisionTreeClassifier(random_state=42)),
        ],
        n_jobs=-1,
        passthrough=True,
        verbose=1,
         )
param_dist11 = {
       'passthrough': [True],
        'final_estimator': [LogisticRegression(random_state=42),  ('rf', RandomForestClassifier(random_state=42)),
            ('dt', DecisionTreeClassifier(random_state=42))]
}

In [ ]:
from sklearn.model_selection import RandomizedSearchCV
# StackingClassifier

random_search_stacking = RandomizedSearchCV(model_stacking, param_distributions=param_dist11, n_iter=500,
                                   cv=10, scoring="accuracy", random_state=42, n_jobs=-1)
random_search_stacking.fit(X_train, y_train)

best_model_stacking = random_search_stacking.best_estimator_
print("Best parameters:", random_search_stacking.best_params_)
print("Best CV accuracy: {:.4f}".format(random_search_stacking.best_score_))
print("Test accuracy: {:.4f}".format(best_model_stacking.score(X_test, y_test)))

Best parameters: {'passthrough': True, 'final_estimator': LogisticRegression(random_state=42)}
Best CV accuracy: 0.8575
Test accuracy: 0.8970


In [ ]:

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, matthews_corrcoef,
    cohen_kappa_score, balanced_accuracy_score
)
import numpy as np
import pandas as pd
import time

# Define your classifier here (StackingClassifier)
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier

clf_name = "StackingClassifier"
clf_stacking = best_model_stacking # Use the best model from RandomizedSearchCV


# Store results
results_stacking = []



# KFold Cross-Validation
from sklearn.model_selection import KFold
kf = KFold(n_splits=10, shuffle=True, random_state=42)

for fold, (train_idx, test_idx) in enumerate(kf.split(X_shuffled)):
    X_train, X_test = X_shuffled.iloc[train_idx], X_shuffled.iloc[test_idx]
    y_train, y_test = y_shuffled.iloc[train_idx], y_shuffled.iloc[test_idx]

    # Train
    start_time = time.time()
    clf_stacking.fit(X_train, y_train)
    training_time = time.time() - start_time

    # Predict
    y_pred = clf_stacking.predict(X_test)

    # Metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    cm = confusion_matrix(y_test, y_pred)
    tn, fp, fn, tp = cm.ravel()
    specificity = tn / (tn + fp) if (tn + fp) != 0 else 0
    fpr = fp / (fp + tn) if (fp + tn) != 0 else 0
    gm = np.sqrt(rec * specificity)
    auc = roc_auc_score(y_test, y_pred)
    mcc = matthews_corrcoef(y_test, y_pred)
    kappa = cohen_kappa_score(y_test, y_pred)
    balanced_acc = balanced_accuracy_score(y_test, y_pred)

    # Store fold results
    results_stacking.append({
        "Fold": fold+1,
        "Classifier": clf_name,
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "Specificity": specificity,
        "F1": f1,
        "GM": gm,
        "FPR": fpr,
        "AUC": auc,
        "MCC": mcc,
        "Kappa": kappa,
        "Balanced Accuracy": balanced_acc,
        "Training Time (s)": training_time
    })

# Create DataFrame
results_stacking_df= pd.DataFrame(results_stacking)

# Print
print(f"Results for {clf_name}:")
print(results_stacking_df)

# Save to CSV if needed
results_stacking_df.to_csv(f"{clf_name.replace(' ', '_')}_metrics.csv", index=False)

Results for StackingClassifier:
   Fold          Classifier  Accuracy  Precision    Recall  Specificity  \
0     1  StackingClassifier  0.758679   0.226337  0.476190     0.794367   
1     2  StackingClassifier  0.731488   0.208295  0.477801     0.764399   
2     3  StackingClassifier  0.901918   0.656250  0.231278     0.984993   
3     4  StackingClassifier  0.767176   0.240838  0.495690     0.801642   
4     5  StackingClassifier  0.752124   0.203782  0.424508     0.793009   
5     6  StackingClassifier  0.754552   0.236760  0.452381     0.796680   
6     7  StackingClassifier  0.750910   0.201403  0.467442     0.783952   
7     8  StackingClassifier  0.748725   0.216972  0.502232     0.778807   
8     9  StackingClassifier  0.761292   0.231088  0.480603     0.796935   
9    10  StackingClassifier  0.897037   0.738095  0.192149     0.990919   

         F1        GM       FPR       AUC       MCC     Kappa  \
0  0.306834  0.615037  0.205633  0.635279  0.201077  0.182535   
1  0.290116 

In [ ]:
# Combine all results into ONE Excel file (multiple sheets)

with pd.ExcelWriter("smote.xlsx") as writer:

    results_df.to_excel(writer, sheet_name="Logistic Regression", index=False)
    results1_df1.to_excel(writer, sheet_name="Decision Tree", index=False)
    results2_df2.to_excel(writer, sheet_name="Random Forest", index=False)
    results3_df3.to_excel(writer, sheet_name="KNN", index=False)
    results4_df4.to_excel(writer, sheet_name="SVM", index=False)
    results5_df5.to_excel(writer, sheet_name="XGBoost", index=False)
    results6_df6.to_excel(writer, sheet_name="LightGBM", index=False)
    results7_df7.to_excel(writer, sheet_name="CatBoost", index=False)
    results8_df8.to_excel(writer, sheet_name="Gradient Boosting", index=False)
    results9_df9.to_excel(writer, sheet_name="MLP", index=False)
    results_bagging_df.to_excel(writer, sheet_name="Bagging", index=False)
    results_stacking_df.to_excel(writer, sheet_name="Stacking", index=False)


print("Excel file created successfully!")

Excel file created successfully!


In [ ]:
from google.colab import files
files.download("smote.xlsx")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>